# FASTQ quality control

Portable upstream QC for the AOP-guided toxicogenomics pipeline. The notebook discovers local `*.fastq.gz`/`*.fq.gz` files, runs FastQC, optionally aggregates reports with MultiQC, and exports machine-readable module results. Configure paths through `AOP_FASTQ_DIR` and `AOP_FASTQC_OUTPUT_DIR`; no data are downloaded and no software is installed by the notebook.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import zipfile

FASTQ_DIR = Path(os.environ.get('AOP_FASTQ_DIR', 'data/fastq')).expanduser().resolve()
OUT_DIR = Path(os.environ.get('AOP_FASTQC_OUTPUT_DIR', 'outputs/FastQC')).expanduser().resolve()
THREADS = max(1, int(os.environ.get('AOP_FASTQC_THREADS', '4')))
OUT_DIR.mkdir(parents=True, exist_ok=True)

patterns = ('*.fastq.gz', '*.fq.gz', '*.fastq', '*.fq')
fastq_files = sorted({p for pattern in patterns for p in FASTQ_DIR.glob(pattern)})
if not fastq_files:
    raise FileNotFoundError(f'No FASTQ files found in {FASTQ_DIR}')
print(f'FASTQ directory: {FASTQ_DIR}')
print(f'Files discovered: {len(fastq_files)}')
print(f'Output directory: {OUT_DIR}')

In [ ]:
fastqc = shutil.which('fastqc')
if fastqc is None:
    raise RuntimeError('FastQC is not on PATH. Install FastQC before running this optional stage.')

command = [fastqc, '--threads', str(THREADS), '--outdir', str(OUT_DIR), *map(str, fastq_files)]
print('Running:', ' '.join(command[:6]), f'... ({len(fastq_files)} files)')
subprocess.run(command, check=True)

In [ ]:
multiqc = shutil.which('multiqc')
if multiqc is None:
    print('MultiQC is not on PATH; individual FastQC reports were retained.')
else:
    subprocess.run([multiqc, str(OUT_DIR), '--outdir', str(OUT_DIR), '--force'], check=True)

In [ ]:
module_rows = []
for archive in sorted(OUT_DIR.glob('*_fastqc.zip')):
    with zipfile.ZipFile(archive) as zf:
        summary_name = next((name for name in zf.namelist() if name.endswith('/summary.txt')), None)
        if summary_name is None:
            continue
        for line in zf.read(summary_name).decode('utf-8', errors='replace').splitlines():
            status, module, source_file = line.split('\t', 2)
            module_rows.append({'sample': source_file, 'module': module, 'status': status})

summary_csv = OUT_DIR / 'fastqc_module_summary.csv'
with summary_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['sample', 'module', 'status'])
    writer.writeheader()
    writer.writerows(module_rows)
print(f'Wrote {summary_csv} ({len(module_rows)} module assessments)')

In [ ]:
manifest = {
    'fastq_directory': str(FASTQ_DIR),
    'output_directory': str(OUT_DIR),
    'file_count': len(fastq_files),
    'files': [str(path) for path in fastq_files],
    'fastqc_executable': fastqc,
    'fastqc_version': subprocess.run([fastqc, '--version'], capture_output=True, text=True, check=True).stdout.strip(),
    'multiqc_executable': multiqc,
    'threads': THREADS,
}
manifest_path = OUT_DIR / 'fastqc_run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Wrote {manifest_path}')